# 5 Gate, highway, or route: one idea from three papers

`04_composition` closed on a routing experiment: Gate, Highway, MoE all made a
four-patch window *worse*. That was an honest result, not a verdict on the family --
four patches of a smooth univariate series do not have much for a router to route
between. This notebook starts over, on ground the family is actually built for, and
traces the mechanism to where it comes from rather than presenting it as free-floating.

**This is not a time-series notebook.** The gate/highway/MoE family composes with any
`(batch, hidden)` tensor -- images, text, tabular data, anything. It sits here because
this lesson already has a running example and a baseline table to beat, the same reason
`1_pytorch_intro/04_search` introduced ray.tune where a two-second training run made it
obviously worth the investment rather than because tuning is somehow a time-series idea.


## 5.1 One idea, three resolutions

| component | mechanism | where it comes from |
|---|---|---|
| `Gate` | `sigmoid(Wx) * x` -- a learned per-dimension scale | the gate inside every LSTM cell (Hochreiter & Schmidhuber, 1997) and the reset/update gates of the GRU (Cho et al., 2014, *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*) |
| `Highway` | `g * T(x) + (1 - g) * x` -- a learned interpolation between a transform and the identity | Highway Networks (Srivastava, Greff & Schmidhuber, 2015), built explicitly to let very deep plain networks train at all -- the direct ancestor of the ResNet residual connection, which is a Highway with the gate fixed at 0.5 |
| `MoE` | `sum_i softmax(Wx)_i * expert_i(x)` -- N gates routing between N experts | Jacobs & Jordan, 1991, *Adaptive Mixtures of Local Experts* -- the dense version this notebook uses. Modern *sparse* MoE (Shazeer et al., 2017, *Outrageously Large Neural Networks*; the Switch Transformer, Fedus et al., 2021) routes each token to only a few of thousands of experts, which is the difference between this cell and a frontier language model, not a difference in the underlying idea |

Read top to bottom, each row is the row above with more capacity to choose from: one
gate choosing between a transform and nothing (`Gate`), one gate choosing between a
transform and the identity (`Highway`), N gates choosing between N transforms (`MoE`).

One callback: `1_pytorch_intro/04_search` measured that concatenating two encoder
streams is a wash once the streams are the same width (main_principles.md §3.4c, D10).
Gating and routing are a *different* way two streams -- or two computations on one
stream -- can combine. Whether that earns its parameters here is exactly as much a
measured question as concatenation was there. Predict before you look at the table.


In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from ett import load_ett, window_index
from notebooktester import param
from vectormesh.components import Gate, Highway, MoE, NeuralNet

W, H = 96, 24          # same window as 02_basemodels -- results are directly comparable
HIDDEN = 64
EPOCHS = param(10, test=1)
SEEDS = (0, 1, 2)

ett = load_ett()
mu, sigma = ett.values[ett.splits["train"]].mean(), ett.values[ett.splits["train"]].std()


def windows(split: str, stride: int = 1):
    values = (ett.values[ett.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train")
xva, yva = windows("val")
xte, yte = windows("test")
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}   test {tuple(xte.shape)}")


train (8521, 96)   val (2761, 96)   test (2761, 96)


## 5.2 The baselines to beat

Same discipline as `02_basemodels`: name the free baseline and the cheapest trained one
before anything with a gate in it gets to run.


In [2]:
def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def fit(make_model, seeds=SEEDS, epochs=EPOCHS, batch_size=64, lr=1e-3):
    scores, secs = [], []
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
        t0 = time.time()
        for _ in range(epochs):
            for xb, yb in loader:
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
        secs.append(time.time() - t0)
        model.eval()
        with torch.no_grad():
            scores.append(mae(yva, model(xva)))
    return {
        "mae": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "params": sum(p.numel() for p in model.parameters()),
        "secs": float(np.mean(secs)),
    }


results: dict[str, dict] = {}


def report(name: str, r: dict) -> None:
    results[name] = r
    print(f"{name:32s} MAE {r['mae']:.4f} +/- {r['std']:.4f}   {r['params']:>7,} params   {r['secs']:5.1f}s")


naive = xva[:, -1:].expand(-1, H)
results["naive (repeat last)"] = {"mae": mae(yva, naive), "std": 0.0, "params": 0, "secs": 0.0}
print(f"{'naive (repeat last)':32s} MAE {results['naive (repeat last)']['mae']:.4f}")


class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W, H)

    def forward(self, x):
        return self.fc(x)


report("linear", fit(Linear))


class PlainMLP(nn.Module):
    # NeuralNet([W, HIDDEN], H) = Linear(W, HIDDEN) -> GELU -> Linear(HIDDEN, H).
    # This is the width every gated block below is built on top of.
    def __init__(self):
        super().__init__()
        self.net = NeuralNet(hidden_size=[W, HIDDEN], out_size=H)

    def forward(self, x):
        return self.net(x)


report("plain MLP, no gate", fit(PlainMLP))


naive (repeat last)              MAE 0.1962


linear                           MAE 0.1744 +/- 0.0004     2,328 params     0.2s


plain MLP, no gate               MAE 0.1724 +/- 0.0014     7,768 params     0.3s


## 5.3 Gate: a learned per-dimension scale

`Gate(hidden_size)` computes `sigmoid(Wx) * x` -- a value in the hidden representation
either passes through close to unchanged or gets scaled toward zero, and the network
decides which per dimension, per example. Wrap it around the same `HIDDEN`-wide layer
`PlainMLP` used, so the only thing that changed is the gate.


In [3]:
class GatedBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(W, HIDDEN)
        self.act = nn.GELU()
        self.gate = Gate(hidden_size=HIDDEN)
        self.out = nn.Linear(HIDDEN, H)

    def forward(self, x):
        z = self.act(self.inp(x))
        z = self.gate(z)
        return self.out(z)


report("+ Gate", fit(GatedBlock))


+ Gate                           MAE 0.1715 +/- 0.0006    11,928 params     0.4s


## 5.4 Highway: interpolating between a transform and the identity

`Highway(transform, hidden_size)` computes `g * T(x) + (1 - g) * x`: a second learned
gate `g`, but now it blends between a full transform `T` and doing nothing at all,
rather than between a value and zero. `T` is a `NeuralNet(HIDDEN, HIDDEN)` -- the same
family of block `02_basemodels` and `04_composition` have both used all lesson.


In [4]:
class HighwayBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(W, HIDDEN)
        self.act = nn.GELU()
        self.highway = Highway(transform=NeuralNet(HIDDEN, HIDDEN), hidden_size=HIDDEN)
        self.out = nn.Linear(HIDDEN, H)

    def forward(self, x):
        z = self.act(self.inp(x))
        z = self.highway(z)
        return self.out(z)


report("+ Highway", fit(HighwayBlock))


+ Highway                        MAE 0.1871 +/- 0.0016    20,376 params     0.8s


## 5.5 MoE: many small transforms, one router

`MoE(experts, hidden_size, out_size)` replaces the single gate with a softmax **router**
over N experts, and blends all N outputs by the router's weights -- dense, not sparse:
every expert runs on every example, which is exactly right at four experts and exactly
wrong at four thousand (that gap is the whole reason sparse MoE exists in production
language models). Four linear experts here, same `HIDDEN` width throughout.


In [5]:
class MoEBlock(nn.Module):
    def __init__(self, n_experts: int = 4):
        super().__init__()
        self.inp = nn.Linear(W, HIDDEN)
        self.act = nn.GELU()
        self.moe = MoE(
            experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
            hidden_size=HIDDEN, out_size=HIDDEN,
        )
        self.out = nn.Linear(HIDDEN, H)

    def forward(self, x):
        z = self.act(self.inp(x))
        z = self.moe(z)
        return self.out(z)


report("+ MoE (4 experts)", fit(MoEBlock))


+ MoE (4 experts)                MAE 0.1730 +/- 0.0006    24,668 params     0.8s


## 5.6 The capacity-matched control

Every block above has more parameters than `plain MLP, no gate` -- a gate, a highway
transform, or four experts are all extra weights. Before crediting the *idea*, rule out
that the win (if there is one) is just "more parameters," the same discipline
`1_pytorch_intro` notebook 02 §2.1 and `04_composition` §4.7 both insist on. Add a
second plain hidden layer of the same width, with no gate anywhere -- roughly the same
parameter budget as `+ Highway`, with nothing learned about when to use it.


In [6]:
class DeeperMLP(nn.Module):
    # NeuralNet([W, HIDDEN, HIDDEN], H): one more Linear -> GELU than PlainMLP, no gate.
    def __init__(self):
        super().__init__()
        self.net = NeuralNet(hidden_size=[W, HIDDEN, HIDDEN], out_size=H)

    def forward(self, x):
        return self.net(x)


report("deeper MLP, same budget, no gate", fit(DeeperMLP))


deeper MLP, same budget, no gate MAE 0.1716 +/- 0.0017    11,928 params     0.4s


## 5.7 The table

In [7]:
order = sorted(results, key=lambda k: results[k]["mae"])
print(f"{'model':32s} {'val MAE':>9} {'+/-':>7} {'vs naive':>9} {'params':>9} {'secs':>7}")
print("-" * 78)
base = results["naive (repeat last)"]["mae"]
for name in order:
    r = results[name]
    print(f"{name:32s} {r['mae']:9.4f} {r['std']:7.4f} {r['mae'] / base:9.3f} "
          f"{r['params']:9,} {r['secs']:7.1f}")


model                              val MAE     +/-  vs naive    params    secs
------------------------------------------------------------------------------
+ Gate                              0.1715  0.0006     0.874    11,928     0.4
deeper MLP, same budget, no gate    0.1716  0.0017     0.875    11,928     0.4
plain MLP, no gate                  0.1724  0.0014     0.879     7,768     0.3
+ MoE (4 experts)                   0.1730  0.0006     0.882    24,668     0.8
linear                              0.1744  0.0004     0.889     2,328     0.2
+ Highway                           0.1871  0.0016     0.954    20,376     0.8
naive (repeat last)                 0.1962  0.0000     1.000         0     0.0


Measured, `EPOCHS=10`, 3 seeds:

| model | val MAE | params | vs its capacity-matched control |
|---|---|---|---|
| `+ Gate` | **0.1715** | 11,928 | ties `deeper MLP, same budget, no gate` (0.1716, identical param count) |
| `deeper MLP, no gate` | 0.1716 | 11,928 | -- |
| `plain MLP, no gate` | 0.1724 | 7,768 | -- |
| `+ MoE (4 experts)` | 0.1730 | 24,668 | worse than `plain MLP` at 3x the parameters |
| `linear` | 0.1744 | 2,328 | -- |
| `+ Highway` | **0.1871** | 20,376 | worse than `linear`, at 9x the parameters |

`Gate` and `deeper MLP, no gate` land on the *same* parameter count by construction (a
gate's weight matrix costs exactly what a second `Linear -> GELU` layer costs at this
width) and score within each other's seed noise. That is the control working exactly as
intended: on this window, the gate mechanism itself bought nothing beyond what the extra
parameters would have bought unconditionally. `MoE` and `Highway` cost more and score
worse -- routing capacity that has nothing to route between is not neutral, it is a
worse-fitting optimisation problem. Same shape of result as `04_composition`'s routing
experiment, reached from ETTh1's plain window instead of a four-patch one: this is not
a dataset-specific fluke, it is the family being a real bet that does not always pay.

## What to take from this

- Gate, Highway and MoE are one idea -- a learned keep-or-replace decision -- at
  increasing resolution, and each one traces to a specific paper rather than floating
  free in a library's component list.
- The mechanism is general-purpose. It showed up here because this lesson had a window
  and a baseline ready, not because time series needs it more than any other modality.
- Whether gating earned its extra parameters *on this window* is the number in the
  table above, not an opinion -- compare each gated row to `deeper MLP, same budget, no
  gate` before crediting the gate itself for anything the extra width would have bought
  on its own.
- `04_composition` §4.6 measured the same family losing on a four-patch window. Two
  different measured outcomes on the same family is not a contradiction to resolve --
  it is exactly the no-free-lunch argument `main_principles.md` §1.4 makes about fusion,
  arrived at a second time from a different direction. `06_search` is where you stop
  guessing which regime you are in and let a search tell you.

## Exercises

**E1 -- predict, then measure.** Before re-running this notebook, write down a ranking
you expect for the five trained models. How much of it survives contact with the table?

**E2 -- turn the gate off.** Force `Gate`'s output to equal its input (patch the
forward pass, or initialise its linear layer to a large positive bias so the sigmoid
saturates near 1) and confirm the block's score converges toward `plain MLP, no gate`'s
-- the collapse argument from `1_pytorch_intro/02_nonlinearity` §2.1, for a gate instead
of a stacked linear layer.

**E3 -- more experts.** Re-run `MoEBlock` with 2, 4, 8 and 16 experts. Does the router
find anything to split on, or does the score just track the parameter count? Plot both.

**E4 -- a different window.** Swap in `pedestrian_counts` (02_basemodels E5 already
loads it) and re-run the whole notebook. Does the ranking change on a series with more
structure for a router to find?
